In [1]:
import os 

In [2]:
%pwd

'/home/ammar/Documents/Learning/MLOps/DS_PROJECT/research'

In [3]:
os.chdir("../")
%pwd

'/home/ammar/Documents/Learning/MLOps/DS_PROJECT'

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir:Path
    source_URL : str
    local_data_file: Path
    unzip_dir: Path
    

In [5]:
from src.DS_PROJECT.constants import *
from src.DS_PROJECT.utils.common import read_yaml,create_directories


In [6]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath=CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH,
                 schema_filepath = SCHEMA_FILE_PATH):
        self.config=read_yaml(config_filepath)
        self.params=read_yaml(params_filepath)
        self.schema=read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])


    def get_data_ingestion_config(self)-> DataIngestionConfig:
        config=self.config.data_ingestion
        create_directories([config.root_dir])

        data_ingestion_config=DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir

        )
        return data_ingestion_config

In [7]:
import os
import urllib.request as request
from src.DS_PROJECT import logger
import zipfile

In [10]:
## component-Data Ingestion

class DataIngestion:
    def __init__(self,config:DataIngestionConfig):
        self.config=config
    
    # Downloading the zip file
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} download! with following info: \n{headers}")
        else:
            logger.info(f"File already exists")

    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)


In [11]:
try:
    config=ConfigurationManager()
    data_ingestion_config=config.get_data_ingestion_config()
    data_ingestion=DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2025-12-04 16:52:08,281]: INFO : common : yaml file: config/config.yaml loaded successfully:
[2025-12-04 16:52:08,282]: INFO : common : yaml file: params.yaml loaded successfully:
[2025-12-04 16:52:08,283]: INFO : common : yaml file: schema.yaml loaded successfully:
[2025-12-04 16:52:08,284]: INFO : common : created directory at: artifacts:
[2025-12-04 16:52:08,284]: INFO : common : created directory at: artifacts/data_ingestion:
[2025-12-04 16:52:11,530]: INFO : 251335794 : artifacts/data_ingestion/data.zip download! with following info: 
Content-Type: application/octet-stream
X-GUploader-UploadID: AOCedOG25BdlMxAPfbMTdiOoaI5uu8Mx6DwxWeKA4cXDwsIRfOUebiBnP4KS5y3JX5BPIQ4mguPYO3s
Content-Security-Policy: sandbox
Content-Security-Policy: default-src 'none'
Content-Security-Policy: frame-ancestors 'none'
X-Content-Security-Policy: sandbox
Cross-Origin-Opener-Policy: same-origin
Cross-Origin-Embedder-Policy: require-corp
Cross-Origin-Resource-Policy: same-site
X-Content-Type-Options: nosni